# Lullaby tempo analyzer (study version)

The notebook used in the lullaby studies, kept as it was run, so results can be
reproduced. Only the folder paths were replaced and one line was made compatible
with newer librosa/NumPy versions.

- **Phase 1** (free singing): broad prior around 60 BPM (40 to 100), tactus
  correction, and the grid-fit octave check for estimates above 95 BPM. This is
  the `lullaby` preset in `vocal_tempo.py`.
- **Phase 2** (experimental conditions): the prior is read from the file name
  (`OT`/`OM` = optimal, about 55 BPM; `FT`/`FM` = fast, about 110 BPM), no grid check.

Set the two folder paths in the last block and run all cells. Folders that do
not exist are skipped. For new projects, `01_quickstart.ipynb` with
`vocal_tempo.py` is easier to adapt.

In [ ]:
# ============================================
# Lullaby Tempo Analyzer (Phase1 & Phase2)
# - Phase 1: adds octave-selection fix (BPM vs BPM/2) by grid-fit
# - Phase 2: original method (no extra octave fix)
# Output: lullaby_tempo_summary_<phase>.csv (+ optional per-file beat CSVs)
# ============================================

import os, re, math
import numpy as np
import pandas as pd
import librosa

# ---------- Core settings ----------
HOP = 512             # ~11.6 ms @ 44.1 kHz
F_PREEMPH = 0.97      # pre-emphasis to lift consonant energy
SMOOTH_FRAMES = 10    # onset smoothing frames
LIBROSA_LOAD_SR = None  # None = keep native SR

# ---------- I/O helpers ----------
def load_preemphasize(path, sr=LIBROSA_LOAD_SR, preemph=F_PREEMPH):
    y, sr = librosa.load(path, sr=sr, mono=True)
    if preemph:
        y = librosa.effects.preemphasis(y, coef=preemph)
    return y, sr

def onset_envelope(y, sr, hop=HOP, smooth_frames=SMOOTH_FRAMES):
    env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop)
    env = librosa.util.normalize(env)
    if smooth_frames and smooth_frames > 1:
        kernel = np.ones(smooth_frames) / smooth_frames
        env = np.convolve(env, kernel, mode='same')
    return env

# ---------- ACF-based periodicity ----------
def acf_unbiased(x):
    N = len(x)
    # positive lags only, skip lag 0
    return np.array([np.dot(x[:N-k], x[k:])/(N-k) for k in range(1, N//2)])

def bpm_axis(sr, hop, length):
    lags = np.arange(1, length+1)  # 1..length
    return 60.0 * sr / (hop * lags)

def acf_pick_tempo(onset_env, sr, hop, tempo_min, tempo_max, mu, sigma=5.0):
    acf = acf_unbiased(onset_env)
    bpms = bpm_axis(sr, hop, len(acf))
    mask = (bpms >= tempo_min) & (bpms <= tempo_max)
    if not np.any(mask):
        return None
    bpms_m = bpms[mask]
    acf_m  = acf[mask]
    prior  = np.exp(-0.5 * ((bpms_m - mu) / sigma)**2)
    score  = acf_m * prior
    return float(bpms_m[np.argmax(score)])

# ---------- Beat tracking + robust BPM ----------
def robust_bpm_from_beats(onset_env, sr, hop, start_bpm):
    bt_bpm, beat_frames = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr,
                                                  hop_length=hop, start_bpm=start_bpm)
    beat_times = librosa.frames_to_time(beat_frames, sr=sr, hop_length=hop)
    if len(beat_times) < 2:
        return dict(bt_bpm=float('nan'), rbt_bpm=float('nan'), beat_times=beat_times)
    ibis = np.diff(beat_times)
    rbt_bpm = 60.0 / np.median(ibis)
    return dict(bt_bpm=float(np.atleast_1d(bt_bpm)[0]), rbt_bpm=float(rbt_bpm), beat_times=beat_times)

# ---------- Priors & tactus correction ----------
def expected_from_filename(fname):
    f = fname.upper()
    if "OT" in f or "OM" in f:
        return dict(cond="Optimal", mu=55,  tmin=40, tmax=70,  start_bpm=55)
    if "FT" in f or "FM" in f:
        return dict(cond="Fast",    mu=110, tmin=90, tmax=130, start_bpm=110)
    return dict(cond="Unknown", mu=80, tmin=40, tmax=150, start_bpm=80)

def phase1_expected_prior():
    # Broad, phase-1 prior (free singing)
    return dict(cond="Phase1", mu=60, tmin=40, tmax=100, start_bpm=60)

def tactus_correction(bpm, mu):
    if bpm is None or (isinstance(bpm, float) and (np.isnan(bpm) or bpm <= 0)):
        return bpm, False
    if abs(bpm - 2*mu) < abs(bpm - mu):
        return bpm/2.0, True
    return bpm, False

# ---------- Phase-1-only octave selection by grid-fit ----------
def grid_fit_score(onset_env, bpm, sr, hop, n_phase=32):
    """How well a BPM grid (with best phase) aligns to onset envelope."""
    if bpm is None or (isinstance(bpm, float) and (np.isnan(bpm) or bpm <= 0)):
        return -np.inf
    period = 60.0 * sr / (hop * bpm)   # frames per beat
    if period < 8:
        return -np.inf
    N = len(onset_env)
    phases = np.linspace(0, period, num=n_phase, endpoint=False)
    best = -np.inf
    for phi in phases:
        idx = (phi + np.arange(0, N, period)).astype(int)
        idx = idx[idx < N]
        if idx.size < 4:
            continue
        s = onset_env[idx].mean()
        if s > best:
            best = s
    return float(best)

def choose_octave_by_fit(onset_env, bpm, sr, hop, prefer_slower_if_close=True, close_margin=0.03):
    """Compare grid-fit at BPM vs BPM/2; pick half if clearly better (or near-tie→slower)."""
    if bpm is None or (isinstance(bpm, float) and np.isnan(bpm)):
        return bpm, False, {"score_x": np.nan, "score_half": np.nan}
    sx  = grid_fit_score(onset_env, bpm,   sr, hop)
    sh  = grid_fit_score(onset_env, bpm/2, sr, hop)
    if sh > sx * (1 + close_margin):  # clearly better at half
        return bpm/2.0, True, {"score_x": sx, "score_half": sh}
    if prefer_slower_if_close and abs(sh - sx) <= abs(sx)*close_margin and (bpm/2.0) >= 40:
        return bpm/2.0, True, {"score_x": sx, "score_half": sh}
    return bpm, False, {"score_x": sx, "score_half": sh}

# ---------- Reporting choice ----------
def pick_reported_bpm(result):
    bpm = result.get("RBT_BPM_final", np.nan)
    if bpm is None or (isinstance(bpm, float) and np.isnan(bpm)):
        bpm = result.get("ACF_BPM_final", np.nan)
    return None if (bpm is None or (isinstance(bpm, float) and np.isnan(bpm))) else round(float(bpm), 0)

# ---------- Single-file analysis ----------
def analyze_file(path, phase="phase2"):
    fname = os.path.basename(path)

    # Priors
    if phase.lower() == "phase1":
        meta = phase1_expected_prior()
    else:
        meta = expected_from_filename(fname)

    # Features
    y, sr = load_preemphasize(path)
    env = onset_envelope(y, sr, hop=HOP, smooth_frames=SMOOTH_FRAMES)

    # ACF + prior
    acf_bpm = acf_pick_tempo(env, sr, HOP, meta['tmin'], meta['tmax'], mu=meta['mu'], sigma=5.0)
    acf_corr, acf_halved = tactus_correction(acf_bpm, meta['mu'])

    # Beat tracker + robust median-IBI BPM
    beats = robust_bpm_from_beats(env, sr, HOP, start_bpm=meta['start_bpm'])
    rbt_corr, rbt_halved = tactus_correction(beats['rbt_bpm'], meta['mu'])

    # Phase-1 ONLY: octave selection by grid-fit for high tempos
    fit_scores = {}
    if phase.lower() == "phase1":
        if (rbt_corr is not None) and not np.isnan(rbt_corr) and rbt_corr > 95:
            rbt_corr2, halved2, scores = choose_octave_by_fit(env, rbt_corr, sr, HOP,
                                                              prefer_slower_if_close=True, close_margin=0.03)
            fit_scores["RBT"] = scores
            rbt_halved = rbt_halved or halved2
            rbt_corr = rbt_corr2
        if (acf_corr is not None) and not np.isnan(acf_corr) and acf_corr > 95:
            acf_corr2, halved2, scores = choose_octave_by_fit(env, acf_corr, sr, HOP,
                                                              prefer_slower_if_close=True, close_margin=0.03)
            fit_scores["ACF"] = scores
            acf_halved = acf_halved or halved2
            acf_corr = acf_corr2

    return {
        "Filename": fname,
        "Phase": phase,
        "Condition": meta['cond'],
        "ACF_BPM_final": acf_corr,
        "ACF_Halved": acf_halved,
        "BeatTracker_Tempo_BPM": beats['bt_bpm'],
        "RBT_BPM_final": rbt_corr,
        "RBT_Halved": rbt_halved,
        "Num_Beats": len(beats['beat_times']),
        "OctaveFitScores": fit_scores,
        "Beat_Times_sec": beats['beat_times']
    }

# ---------- Batch processing ----------
def process_folder(folder, save_beats=True, phase="phase2"):
    rows = []
    halved_count = 0
    for fn in sorted(os.listdir(folder)):
        if not fn.lower().endswith(('.wav', '.mp3', '.flac', '.m4a', '.ogg')):
            continue
        path = os.path.join(folder, fn)
        try:
            res = analyze_file(path, phase=phase)
            res["Reported_BPM"] = pick_reported_bpm(res)
            if res.get("RBT_Halved", False):
                halved_count += 1
            rows.append(res)

            if save_beats and len(res["Beat_Times_sec"]) > 1:
                dfb = pd.DataFrame({
                    "Beat_Index": np.arange(1, len(res["Beat_Times_sec"])+1),
                    "Beat_Time_sec": res["Beat_Times_sec"]
                })
                out_beats = os.path.join(folder, os.path.splitext(fn)[0] + "_beats.csv")
                dfb.to_csv(out_beats, index=False)
        except Exception as e:
            print(f"⚠️ {fn}: {e}")

    if not rows:
        print("No audio files found.")
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    # Compact summary: drop heavy arrays before CSV
    keep_front = ["Filename","Phase","Condition","Reported_BPM",
                  "RBT_BPM_final","BeatTracker_Tempo_BPM","ACF_BPM_final","Num_Beats",
                  "RBT_Halved","ACF_Halved","OctaveFitScores"]
    keep_front = [c for c in keep_front if c in df.columns]
    other_cols = [c for c in df.columns if c not in keep_front and c != "Beat_Times_sec"]
    df_out = df[keep_front + other_cols]

    out_csv = os.path.join(folder, f"lullaby_tempo_summary_{phase.lower()}.csv")
    df_out.to_csv(out_csv, index=False)
    print(f"✅ Saved: {out_csv}")
    print(f"Octave corrections applied (RBT_Halved=True): {halved_count}")
    return df_out

# ==== RUN THIS BLOCK (edit the paths to yours) ====
import os

# Phase 1 (use the octave-fix)
folder_phase1 = r"path/to/phase1_audio_folder"   # free singing (octave grid check on)

# Phase 2 (leave method unchanged)
folder_phase2 = r"path/to/phase2_audio_folder"   # condition-coded files (OT/OM = optimal, FT/FM = fast); leave as is to skip

# --- sanity checks & logging ---
def count_audio(folder):
    if not os.path.isdir(folder):
        print("❌ Folder not found:", folder)
        return 0
    exts = ('.wav','.mp3','.flac','.m4a','.ogg')
    n = sum(fn.lower().endswith(exts) for fn in os.listdir(folder))
    print(f"📂 {folder}\n   → audio files found: {n}")
    return n

print("🔧 Environment OK, starting …")
n1 = count_audio(folder_phase1)
if n1 > 0:
    print("▶️ Processing PHASE 1 …")
    df1 = process_folder(folder_phase1, save_beats=True, phase="phase1")
    display(df1.head())

n2 = count_audio(folder_phase2)
if n2 > 0:
    print("▶️ Processing PHASE 2 …")
    df2 = process_folder(folder_phase2, save_beats=True, phase="phase2")
    display(df2.head())

print("✅ Done.")